# Shape-Space Velocity Comparison, Part 1

This notebook compares the best saved old subset `64/128/64`, new baseline-aligned subset `64/160/32`, and additive-best models. It focuses on checkpoint selection, test-time disease prediction for `64/160/32`, real-data speed references, mean-anchor component speed, shape-space Jacobian maps, and `INR(mean_z) - INR(mean_z + v*t)` finite-step maps.


In [1]:
# This cell defines the comparison scope and loads all common libraries.
# The notebook compares three best saved models: old subset 64/128/64,
# new baseline-aligned subset 64/160/32, and additive strong best.
import json
import html as html_lib
import math
import random
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import trimesh
from IPython.display import display, Markdown, HTML
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from skimage.measure import marching_cubes

ROOT = Path('/home/jakaria/INR/Deep3DComp')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from deep_sdf import data as deep_sdf_data
from networks.deep_sdf_decoder import Decoder
from networks.longitudinal_disentangled_flow_64_128_64_adv import (
    build_temporal_flow as build_subset128_flow,
)
from networks.longitudinal_flow_64_160_32_pred_dx import (
    build_temporal_flow as build_subset160_flow,
)
from networks.longitudinal_additive_flow import (
    build_temporal_flow as build_additive_flow,
)

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

GPU_ID = 0
if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)
DEVICE = torch.device(f'cuda:{GPU_ID}' if torch.cuda.is_available() else 'cpu')

# Geometry and derivative settings. Increase GRID_RES only after the notebook runs once.
GRID_RES = 64
GRID_BATCH = 2**17
SURFACE_BATCH = 4096
DELTA_T = 0.10
FD_EPS = 1e-3
EVAL_TIME = 0.50
DIAGNOSIS_FOR_COMPONENT_MAPS = 1.0
COMPONENTS = ('age', 'disease_raw', 'disease', 'residual', 'total')

NOTEBOOK_TITLE = 'Shape-Space Velocity Comparison Part 1'
OUTPUT_DIR = ROOT / 'analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1'
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Match the previous HTML-report notebooks: save heavy figures externally and keep
# notebook outputs light. Run the final index cell after figure/table cells finish.
SAVE_INTERACTIVE_HTML = True
SHOW_INLINE_PLOTLY = False
SHOW_INLINE_MATPLOTLIB = False
FIGURE_MANIFEST = {}


def figure_slug(value):
    text = re.sub(r'[^A-Za-z0-9._-]+', '_', str(value)).strip('_')
    return text or 'output'


def register_output(path, title, category, description=''):
    path = Path(path)
    FIGURE_MANIFEST[str(path)] = {
        'path': path,
        'title': str(title),
        'category': str(category),
        'description': str(description),
    }
    print('Saved output:', path)


def html_page(title, body, description=''):
    desc = f"<p class='note'>{html_lib.escape(str(description))}</p>" if description else ''
    return f"""<!doctype html>
<html><head><meta charset='utf-8'><title>{html_lib.escape(str(title))}</title>
<style>
body{{font-family:Arial,sans-serif;max-width:1200px;margin:32px auto;padding:0 22px;color:#202020}}
h1{{margin-bottom:8px}} .note{{background:#edf6f5;border-left:4px solid #147d92;padding:12px 14px;border-radius:6px}}
table{{border-collapse:collapse;width:100%;font-size:13px}} th,td{{border:1px solid #ddd;padding:7px 9px;text-align:left}}
th{{background:#f2eadc}} img{{max-width:100%;border:1px solid #ddd;border-radius:8px}} code{{background:#f3f3f3;padding:2px 5px;border-radius:3px}}
a{{color:#0b5cad;text-decoration:none}} a:hover{{text-decoration:underline}}
</style></head><body><h1>{html_lib.escape(str(title))}</h1>{desc}{body}</body></html>"""


def save_table(df, filename, title, category, description=''):
    stem = figure_slug(filename)
    html_path = FIGURE_DIR / f'{stem}.html'
    csv_path = FIGURE_DIR / f'{stem}.csv'
    df.to_csv(csv_path, index=False)
    body = (
        f"<p><strong>CSV:</strong> <a href='{csv_path.name}'>{csv_path.name}</a></p>"
        + df.to_html(index=False, escape=False)
    )
    html_path.write_text(html_page(title, body, description), encoding='utf-8')
    register_output(html_path, title, category, description)
    return html_path


def save_plotly_figure(fig, filename, title, category, description='', show_inline=False):
    html_path = FIGURE_DIR / f'{figure_slug(filename)}.html'
    if SAVE_INTERACTIVE_HTML:
        fig.write_html(html_path, include_plotlyjs='directory', full_html=True, auto_open=False)
        register_output(html_path, title, category, description)
    if SHOW_INLINE_PLOTLY and show_inline:
        fig.show()
    return html_path


def save_matplotlib_figure(fig, filename, title, category, description='', dpi=170):
    stem = figure_slug(filename)
    png_path = FIGURE_DIR / f'{stem}.png'
    html_path = FIGURE_DIR / f'{stem}.html'
    fig.savefig(png_path, dpi=dpi, bbox_inches='tight', facecolor='white')
    body = f"<p><img src='{png_path.name}' alt='{html_lib.escape(str(title))}'></p>"
    html_path.write_text(html_page(title, body, description), encoding='utf-8')
    register_output(html_path, title, category, description)
    if SHOW_INLINE_MATPLOTLIB:
        plt.show()
    else:
        plt.close(fig)
    return html_path


def write_figure_index(report_title=NOTEBOOK_TITLE):
    entries = sorted(
        FIGURE_MANIFEST.values(),
        key=lambda item: (item['category'], item['title'], str(item['path'])),
    )
    groups = {}
    for item in entries:
        groups.setdefault(item['category'], []).append(item)
    chunks = [
        "<!doctype html><html><head><meta charset='utf-8'>",
        f"<title>{html_lib.escape(report_title)}</title>",
        "<style>body{font-family:Arial,sans-serif;max-width:1150px;margin:32px auto;padding:0 20px;color:#202020}",
        "h1{margin-bottom:8px}h2{margin-top:30px}li{margin:10px 0}a{color:#0b5cad;text-decoration:none}",
        "a:hover{text-decoration:underline}.desc{color:#555;font-size:14px}</style></head><body>",
        f"<h1>{html_lib.escape(report_title)}</h1>",
        "<p>All heavy notebook outputs are saved here as HTML pages. Open Plotly pages in a browser for interactive 3D inspection.</p>",
        f"<p><strong>Output directory:</strong> <code>{html_lib.escape(str(FIGURE_DIR))}</code></p>",
    ]
    for category, items in groups.items():
        chunks.append(f"<h2>{html_lib.escape(category)}</h2><ul>")
        for item in items:
            relative = item['path'].relative_to(FIGURE_DIR)
            desc = html_lib.escape(item.get('description', ''))
            chunks.append(
                f"<li><a href='{relative.as_posix()}'>{html_lib.escape(item['title'])}</a>"
                f"<br><span class='desc'>{desc}</span></li>"
            )
        chunks.append('</ul>')
    chunks.append('</body></html>')
    index_path = FIGURE_DIR / 'index.html'
    index_path.write_text(''.join(chunks), encoding='utf-8')
    print('Figure index:', index_path)
    return index_path


BASE = ROOT / 'examples' / 'Torus_subset_100_id_age_progression'
SUBSET128_DIR = BASE / 'longitudinal_age_disease_conditioned_cocycle_shape_pair_loss_multiple_pairs_disentangled_velocity_64_128_64_binary_margin_adv'
SUBSET160_DIR = BASE / 'velocity_64_160_32_baseline_dx'
ADDITIVE_DIR = BASE / 'longitudinal_age_disease_additive_velocity_disentanglement_strong_residual_invariance_adv'

# Best saved checkpoints based on the results currently present:
# - 64/128/64: epoch 500 is the best old GT-forecast checkpoint.
# - 64/160/32: epoch 1000 is the best saved E2 forecast checkpoint; epoch 1500 had better disease CSV but no saved model checkpoint.
# - additive: epoch 1000 is the additive strong best checkpoint used in prior notebooks.
MODEL_CONFIGS = {
    'subset64_128_best': {
        'label': 'Subset 64/128/64 best',
        'short_label': '64/128/64',
        'kind': 'subset128',
        'exp_dir': SUBSET128_DIR,
        'checkpoint': '500',
        'reported_test_chamfer': 0.00014264255878515542,
        'selection_note': 'Best old subset checkpoint from GT ranking.',
    },
    'subset64_160_best_saved': {
        'label': 'Subset 64/160/32 E2 best saved',
        'short_label': '64/160/32',
        'kind': 'subset160',
        'exp_dir': SUBSET160_DIR,
        'checkpoint': '1000',
        'reported_test_chamfer': 0.00018364414945436707,
        'selection_note': 'Best saved baseline-aligned forecast checkpoint; epoch 1500 diagnosis CSV exists but no checkpoint file exists.',
    },
    'additive_best': {
        'label': 'Additive strong best',
        'short_label': 'additive',
        'kind': 'additive',
        'exp_dir': ADDITIVE_DIR,
        'checkpoint': '1000',
        'reported_test_chamfer': 0.00021775640198029578,
        'selection_note': 'Best additive strong checkpoint from prior comparison.',
    },
}

print('Device:', DEVICE)
print('Grid resolution:', GRID_RES)
print('Finite-step interval DELTA_T:', DELTA_T)
print('Evaluation time:', EVAL_TIME, '| forced disease condition for component maps:', DIAGNOSIS_FOR_COMPONENT_MAPS)


Device: cuda:0
Grid resolution: 64
Finite-step interval DELTA_T: 0.1
Evaluation time: 0.5 | forced disease condition for component maps: 1.0


## Load Best Saved Checkpoints

The table explicitly states which checkpoint is loaded and why. The new `64/160/32` model uses the best saved checkpoint, not the best CSV-only epoch.

In [2]:
# This cell loads checkpoint-specific decoders, temporal flows, and mean training anchors.
# The mean anchor is used only for population-level shape-space velocity maps.
def checkpoint_path(exp_dir, subdir, checkpoint):
    name = str(checkpoint)
    if not name.endswith('.pth'):
        name += '.pth'
    path = Path(exp_dir) / subdir / name
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def strip_module_prefix(state):
    return {k.removeprefix('module.'): v for k, v in state.items()}


def latent_weights(payload):
    lat = payload.get('latent_codes_state_dict', payload.get('latent_codes'))
    if isinstance(lat, dict):
        lat = lat['weight']
    if lat.ndim == 3 and lat.shape[1] == 1:
        lat = lat[:, 0]
    return lat.detach().float()


def flow_builder(kind):
    if kind == 'subset128':
        return build_subset128_flow
    if kind == 'subset160':
        return build_subset160_flow
    if kind == 'additive':
        return build_additive_flow
    raise ValueError(f'Unknown model kind: {kind}')


def load_model(config):
    exp_dir = Path(config['exp_dir'])
    specs = json.loads((exp_dir / 'specs.json').read_text())
    decoder = Decoder(int(specs['CodeLength']), **specs['NetworkSpecs']).to(DEVICE)
    flow = flow_builder(config['kind'])(
        specs,
        int(specs['CodeLength']),
        list(specs.get('FlowHiddenDims', [256, 256])),
        age_condition_dim=int(specs.get('AgeConditionDim', 0)),
    ).to(DEVICE)

    model_payload = torch.load(
        checkpoint_path(exp_dir, 'ModelParameters', config['checkpoint']),
        map_location=DEVICE,
    )
    decoder.load_state_dict(strip_module_prefix(model_payload['model_state_dict']))
    flow.load_state_dict(strip_module_prefix(model_payload['flow_state_dict']))
    decoder.eval()
    flow.eval()

    latent_payload = torch.load(
        checkpoint_path(exp_dir, 'LatentCodes', config['checkpoint']),
        map_location='cpu',
    )
    weights = latent_weights(latent_payload)
    mean_z = weights.mean(dim=0, keepdim=True).to(DEVICE)

    return {
        **config,
        'specs': specs,
        'decoder': decoder,
        'flow': flow,
        'mean_z': mean_z,
        'latent_weights': weights,
        'epoch': int(model_payload.get('epoch', -1)),
    }


MODELS = {name: load_model(config) for name, config in MODEL_CONFIGS.items()}

checkpoint_table = pd.DataFrame([
    {
        'model_key': name,
        'model': model['label'],
        'checkpoint': model['checkpoint'],
        'loaded_epoch': model['epoch'],
        'velocity_blocks': (
            model['specs'].get('VelocityAgeDim'),
            model['specs'].get('VelocityDiseaseDim'),
            model['specs'].get('VelocityResidualDim'),
        ),
        'reported_or_current_test_chamfer': model['reported_test_chamfer'],
        'mean_z_norm': float(model['mean_z'].norm().item()),
        'selection_note': model['selection_note'],
    }
    for name, model in MODELS.items()
])
save_table(
    checkpoint_table,
    '01_checkpoint_selection',
    'Checkpoint selection',
    'Model setup',
    'Loaded checkpoints, velocity block dimensions, mean-anchor norms, and selection notes.',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/01_checkpoint_selection.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/01_checkpoint_selection.html')

## Test-Time Disease Prediction for `64/160/32`

This section uses the saved diagnosis-prediction CSVs. The disease label is used only for scoring after the model predicts from the optimized baseline anchor.

In [3]:
# This cell summarizes test-time disease inference for the 64/160/32 model.
# Only the 64/160/32 E2 model has an anchor disease head, so old subset/additive are marked as supervised/oracle for disease condition.
def auc_score(y, score):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    pos = np.flatnonzero(y == 1)
    neg = np.flatnonzero(y == 0)
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    order = np.argsort(score)
    ranks = np.empty(len(score), dtype=float)
    i = 0
    while i < len(score):
        j = i + 1
        while j < len(score) and score[order[j]] == score[order[i]]:
            j += 1
        ranks[order[i:j]] = 0.5 * (i + 1 + j)
        i = j
    return float((ranks[pos].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))


def prediction_metrics_from_csv(path):
    rows = list(csv.DictReader(open(path, newline='')))
    y = np.array([int(float(r['true_diagnosis'])) for r in rows])
    p = np.array([float(r['predicted_probability']) for r in rows])
    pred = np.array([int(float(r['predicted_class'])) for r in rows])
    chamfer = np.array([
        float(r['forecast_mean_chamfer']) for r in rows
        if r.get('forecast_mean_chamfer') not in ('', None)
    ])
    final = np.array([
        float(r['forecast_final_chamfer']) for r in rows
        if r.get('forecast_final_chamfer') not in ('', None)
    ])
    sens = np.mean(pred[y == 1] == 1) if np.any(y == 1) else np.nan
    spec = np.mean(pred[y == 0] == 0) if np.any(y == 0) else np.nan
    epoch = int(re.search(r'epoch_(\d+)', path.name).group(1))
    return {
        'file': path.name,
        'epoch': epoch,
        'n': len(rows),
        'accuracy': float(np.mean(pred == y)),
        'balanced_accuracy': float(0.5 * (sens + spec)),
        'auc': auc_score(y, p),
        'sensitivity': float(sens),
        'specificity': float(spec),
        'forecast_mean_chamfer': float(np.mean(chamfer)) if len(chamfer) else np.nan,
        'forecast_final_chamfer': float(np.mean(final)) if len(final) else np.nan,
        'checkpoint_available': (SUBSET160_DIR / 'ModelParameters' / f'{epoch}.pth').exists(),
        'misclassified_subjects': ', '.join([
            f"{r['subject_id']}({int(float(r['true_diagnosis']))}->{int(float(r['predicted_class']))},p={float(r['predicted_probability']):.2g})"
            for r in rows
            if int(float(r['true_diagnosis'])) != int(float(r['predicted_class']))
        ]),
    }

import csv
pred_dir = SUBSET160_DIR / 'DiagnosisPredictions'
DIAGNOSIS_METRICS = pd.DataFrame([
    prediction_metrics_from_csv(path)
    for path in sorted(pred_dir.glob('test_epoch_*.csv'), key=lambda p: int(re.search(r'epoch_(\d+)', p.name).group(1)))
])
selected_epoch = int(MODELS['subset64_160_best_saved']['epoch'])
DIAGNOSIS_METRICS['selected_for_notebook'] = DIAGNOSIS_METRICS['epoch'].eq(selected_epoch)
save_table(
    DIAGNOSIS_METRICS,
    '02_64_160_disease_prediction_table',
    '64/160/32 test-time disease prediction table',
    'Disease prediction',
    'Disease probability is inferred from the test baseline anchor. Labels are used only for scoring.',
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(DIAGNOSIS_METRICS['epoch'], DIAGNOSIS_METRICS['balanced_accuracy'], marker='o', label='balanced accuracy')
axes[0].plot(DIAGNOSIS_METRICS['epoch'], DIAGNOSIS_METRICS['auc'], marker='s', label='AUC')
axes[0].axvline(selected_epoch, color='black', linestyle='--', alpha=0.6, label='loaded checkpoint')
axes[0].set_title('64/160/32 label-free disease inference')
axes[0].set_xlabel('epoch')
axes[0].set_ylim(0.0, 1.05)
axes[0].grid(alpha=0.25)
axes[0].legend()
axes[1].plot(DIAGNOSIS_METRICS['epoch'], DIAGNOSIS_METRICS['forecast_mean_chamfer'], marker='o', label='mean future Chamfer')
axes[1].plot(DIAGNOSIS_METRICS['epoch'], DIAGNOSIS_METRICS['forecast_final_chamfer'], marker='s', label='final future Chamfer')
axes[1].axvline(selected_epoch, color='black', linestyle='--', alpha=0.6)
axes[1].set_title('64/160/32 label-free forecast error from saved CSV')
axes[1].set_xlabel('epoch')
axes[1].grid(alpha=0.25)
axes[1].legend()
fig.suptitle('Important: epoch 1500 has a CSV but no saved checkpoint; the notebook loads epoch 1000.')
fig.tight_layout()
save_matplotlib_figure(
    fig,
    '03_64_160_disease_prediction_curves',
    '64/160/32 disease prediction and forecast curves',
    'Disease prediction',
    'Balanced accuracy, AUC, and forecast Chamfer from saved diagnosis-prediction CSVs.',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/02_64_160_disease_prediction_table.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/03_64_160_disease_prediction_curves.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/03_64_160_disease_prediction_curves.html')

## Real-Data Speed Reference

Model speeds should be compared with observed torus behavior. Here real speed is computed from consecutive scans using finite differences in thickness and bump height.

In [4]:
# This cell computes observed real-data speeds from torus metadata.
# These are not model outputs. They are finite-difference rates from consecutive scans.
def load_label_dataframe(labels_path):
    obj = torch.load(labels_path, map_location='cpu')
    rows = []
    for scan_id, payload in obj.items():
        row = {'scan_id': str(scan_id)}
        row.update(payload)
        rows.append(row)
    df = pd.DataFrame(rows)
    df['sid'] = df['scan_id'].str.extract(r'ID_(\d+)_t').astype(int)
    df['tp'] = df['scan_id'].str.extract(r'_t(\d+)').astype(int)
    return df.sort_values(['sid', 'tp']).reset_index(drop=True)


def observed_rate_table(label_df):
    rows = []
    for sid, group in label_df.groupby('sid'):
        group = group.sort_values('tp')
        for left, right in zip(group.iloc[:-1].to_dict('records'), group.iloc[1:].to_dict('records')):
            dt_age = float(right['age']) - float(left['age'])
            dt_norm = float(right['age_norm']) - float(left['age_norm'])
            if abs(dt_age) < 1e-8 or abs(dt_norm) < 1e-8:
                continue
            rows.append({
                'sid': sid,
                'diagnosis': int(left['diagnosis']),
                'from_scan': left['scan_id'],
                'to_scan': right['scan_id'],
                'age_mid': 0.5 * (float(left['age']) + float(right['age'])),
                'age_norm_mid': 0.5 * (float(left['age_norm']) + float(right['age_norm'])),
                'dt_years': dt_age,
                'dt_norm': dt_norm,
                'thickness_rate_per_year': (float(right['thickness']) - float(left['thickness'])) / dt_age,
                'bump_rate_per_year': (float(right['bump_height']) - float(left['bump_height'])) / dt_age,
                'thickness_rate_per_norm_time': (float(right['thickness']) - float(left['thickness'])) / dt_norm,
                'bump_rate_per_norm_time': (float(right['bump_height']) - float(left['bump_height'])) / dt_norm,
            })
    return pd.DataFrame(rows)

LABEL_DF = load_label_dataframe(MODELS['subset64_160_best_saved']['specs']['AgeMetadataFile'])
REAL_RATE_DF = observed_rate_table(LABEL_DF)
REAL_RATE_SUMMARY = (
    REAL_RATE_DF.groupby('diagnosis')
    .agg(
        rows=('sid', 'size'),
        thickness_rate_mean=('thickness_rate_per_norm_time', 'mean'),
        thickness_rate_std=('thickness_rate_per_norm_time', 'std'),
        bump_rate_mean=('bump_rate_per_norm_time', 'mean'),
        bump_rate_std=('bump_rate_per_norm_time', 'std'),
    )
    .reset_index()
)
save_table(
    REAL_RATE_SUMMARY,
    '04_real_speed_summary_table',
    'Real-data speed summary',
    'Real speed reference',
    'Observed finite-difference thickness and bump speeds grouped by diagnosis.',
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for diag, color, label in [(0, '#147D92', 'healthy'), (1, '#D1495B', 'diseased')]:
    sub = REAL_RATE_DF[REAL_RATE_DF['diagnosis'] == diag]
    axes[0].scatter(sub['age_mid'], sub['thickness_rate_per_norm_time'], s=18, alpha=0.7, color=color, label=label)
    axes[1].scatter(sub['age_mid'], sub['bump_rate_per_norm_time'], s=18, alpha=0.7, color=color, label=label)
axes[0].axhline(0, color='black', linewidth=1)
axes[1].axhline(0, color='black', linewidth=1)
axes[0].set_title('Observed thickness speed: real finite differences')
axes[1].set_title('Observed bump speed: real finite differences')
for ax in axes:
    ax.set_xlabel('age midpoint')
    ax.grid(alpha=0.25)
    ax.legend()
axes[0].set_ylabel('rate per normalized time')
axes[1].set_ylabel('rate per normalized time')
fig.suptitle('Real-data speed reference. Model speeds below should be interpreted against these trends.')
fig.tight_layout()
save_matplotlib_figure(
    fig,
    '05_real_speed_scatter',
    'Real thickness and bump speeds',
    'Real speed reference',
    'Observed finite-difference speeds from consecutive real torus scans.',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/04_real_speed_summary_table.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/05_real_speed_scatter.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/05_real_speed_scatter.html')

## Geometry and Jacobian Utilities

These functions convert latent velocity into shape-space normal speed using the decoder/INR Jacobian. They also compute finite-difference validation for the Jacobian-vector product.

In [5]:
# This cell defines decoder-Jacobian shape-speed utilities.
# It converts a latent velocity vector v into normal surface speed using the INR Jacobian.
def decode_points(decoder, z, xyz, batch_size=GRID_BATCH):
    z = z.reshape(1, -1)
    outputs = []
    with torch.no_grad():
        for start in range(0, xyz.shape[0], batch_size):
            x = xyz[start:start + batch_size].to(DEVICE)
            zz = z.expand(x.shape[0], -1)
            outputs.append(decoder(torch.cat([zz, x], dim=1)).squeeze(-1).cpu())
    return torch.cat(outputs).numpy()


def regular_grid(resolution=GRID_RES):
    axis = torch.linspace(-1.0, 1.0, int(resolution), dtype=torch.float32)
    grid = torch.stack(torch.meshgrid(axis, axis, axis, indexing='ij'), dim=-1).reshape(-1, 3)
    return axis.numpy(), grid


def decode_grid(decoder, z, resolution=GRID_RES):
    axis, xyz = regular_grid(resolution)
    values = decode_points(decoder, z, xyz).reshape(resolution, resolution, resolution)
    return axis, values


def mesh_from_grid(sdf_grid):
    spacing = 2.0 / (sdf_grid.shape[0] - 1)
    vertices, faces, _, _ = marching_cubes(sdf_grid, level=0.0, spacing=(spacing, spacing, spacing))
    vertices = vertices - 1.0
    mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
    if mesh.is_watertight and mesh.volume < 0.0:
        mesh.invert()
    return mesh


def _zero_pad_subset(model, comp, block):
    flow = model['flow']
    batch = comp.shape[0]
    z = comp.new_zeros(batch, flow.age_dim + flow.disease_dim + flow.residual_dim)
    if block == 'age':
        z[:, :flow.age_dim] = comp
    elif block == 'disease':
        z[:, flow.age_dim:flow.age_dim + flow.disease_dim] = comp
    elif block == 'residual':
        z[:, flow.age_dim + flow.disease_dim:] = comp
    else:
        raise ValueError(block)
    return z


def component_vectors(model, z, time_value=EVAL_TIME, diagnosis=DIAGNOSIS_FOR_COMPONENT_MAPS):
    flow = model['flow']
    t = torch.full((z.shape[0], 1), float(time_value), device=z.device, dtype=z.dtype)
    cond = torch.full((z.shape[0], 1), float(diagnosis), device=z.device, dtype=z.dtype)
    with torch.no_grad():
        comp = flow.velocity_components(z, t, t, age_cond=cond)

    if model['kind'] in ('subset128', 'subset160'):
        age = _zero_pad_subset(model, comp['age'], 'age')
        disease_raw = _zero_pad_subset(model, comp.get('disease_raw', comp['disease']), 'disease')
        disease = _zero_pad_subset(model, comp['disease'], 'disease')
        residual = _zero_pad_subset(model, comp['residual'], 'residual')
    else:
        age = comp['age']
        disease = comp['disease']
        disease_raw = comp.get('disease_raw', disease)
        residual = comp['residual']
    return {
        'age': age,
        'disease_raw': disease_raw,
        'disease': disease,
        'residual': residual,
        'total': age + disease + residual,
    }


def surface_geometry(decoder, z, resolution=GRID_RES):
    axis, grid = decode_grid(decoder, z, resolution)
    mesh = mesh_from_grid(grid)
    vertices = torch.as_tensor(mesh.vertices, dtype=z.dtype, device=DEVICE)

    sdf_parts = []
    grad_parts = []
    for start in range(0, vertices.shape[0], SURFACE_BATCH):
        x = vertices[start:start + SURFACE_BATCH].detach().clone().requires_grad_(True)
        zz = z.detach().expand(x.shape[0], -1)
        sdf = decoder(torch.cat([zz, x], dim=1)).squeeze(-1)
        grad_x = torch.autograd.grad(sdf.sum(), x, create_graph=False)[0]
        sdf_parts.append(sdf.detach().cpu())
        grad_parts.append(grad_x.detach().cpu())

    sdf = torch.cat(sdf_parts).numpy()
    grad_x = torch.cat(grad_parts).numpy()
    grad_norm = np.linalg.norm(grad_x, axis=1) + 1e-12
    grad_unit = grad_x / grad_norm[:, None]
    mesh_normals = np.asarray(mesh.vertex_normals)
    alignment = np.einsum('ij,ij->i', grad_unit, mesh_normals)
    sdf_to_outward_sign = 1.0 if np.nanmedian(alignment) >= 0.0 else -1.0
    outward_normals = sdf_to_outward_sign * grad_unit

    return {
        'axis': axis,
        'sdf_grid': grid,
        'mesh': mesh,
        'vertices': vertices,
        'base_sdf': sdf,
        'grad_x': grad_x,
        'grad_norm': grad_norm,
        'outward_normals': outward_normals,
        'sdf_to_outward_sign': sdf_to_outward_sign,
        'median_normal_alignment': float(np.nanmedian(np.abs(alignment))),
        'signed_mesh_volume': float(mesh.volume),
        'mesh_is_watertight': bool(mesh.is_watertight),
    }


def latent_jvp_at_vertices(decoder, z, velocity, vertices):
    parts = []
    for start in range(0, vertices.shape[0], SURFACE_BATCH):
        x = vertices[start:start + SURFACE_BATCH].detach()
        z0 = z.detach().clone().requires_grad_(True)
        v0 = velocity.detach()
        def decode_for_latent(latent):
            zz = latent.expand(x.shape[0], -1)
            return decoder(torch.cat([zz, x], dim=1)).squeeze(-1)
        _, jvp = torch.autograd.functional.jvp(decode_for_latent, z0, v0, create_graph=False, strict=False)
        parts.append(jvp.detach().cpu())
    return torch.cat(parts).numpy()


def vertex_area_weights(mesh):
    weights = np.zeros(len(mesh.vertices), dtype=np.float64)
    contribution = np.asarray(mesh.area_faces, dtype=np.float64) / 3.0
    for corner in range(3):
        np.add.at(weights, np.asarray(mesh.faces)[:, corner], contribution)
    return weights


def surface_component_map(model, geometry, z, velocity, delta_t=DELTA_T, fd_eps=FD_EPS):
    vertices = geometry['vertices']
    decoder = model['decoder']
    jvp = latent_jvp_at_vertices(decoder, z, velocity, vertices)

    xyz_cpu = vertices.detach().cpu()
    sdf_step = decode_points(decoder, z + delta_t * velocity, xyz_cpu)
    sdf_plus = decode_points(decoder, z + fd_eps * velocity, xyz_cpu)
    sdf_minus = decode_points(decoder, z - fd_eps * velocity, xyz_cpu)
    fd_directional = (sdf_plus - sdf_minus) / (2.0 * fd_eps)

    sign = geometry['sdf_to_outward_sign']
    grad_norm = geometry['grad_norm']
    normal_speed = -sign * jvp / grad_norm
    normal_speed_fd = -sign * fd_directional / grad_norm
    inr_difference = geometry['base_sdf'] - sdf_step
    arrows = normal_speed[:, None] * geometry['outward_normals']

    weights = vertex_area_weights(geometry['mesh'])
    total_area = weights.sum()
    jvp_error = np.sqrt(np.average((jvp - fd_directional) ** 2, weights=weights))
    jvp_scale = np.sqrt(np.average(fd_directional ** 2, weights=weights)) + 1e-12
    metrics = {
        'latent_velocity_norm': float(velocity.norm().item()),
        'mean_abs_normal_speed': float(np.average(np.abs(normal_speed), weights=weights)),
        'rms_normal_speed': float(np.sqrt(np.average(normal_speed ** 2, weights=weights))),
        'net_volume_rate': float(np.sum(normal_speed * weights)),
        'expanding_area_fraction': float(np.sum(weights[normal_speed > 0]) / total_area),
        'contracting_area_fraction': float(np.sum(weights[normal_speed < 0]) / total_area),
        'jvp_fd_relative_rmse': float(jvp_error / jvp_scale),
    }
    return {
        'velocity': velocity.detach(),
        'jvp': jvp,
        'fd_directional': fd_directional,
        'normal_speed': normal_speed,
        'normal_speed_fd': normal_speed_fd,
        'inr_difference': inr_difference,
        'sdf_step': sdf_step,
        'arrows': arrows,
        'metrics': metrics,
    }


def weighted_correlation(a, b, weights):
    w = weights / weights.sum()
    am = np.sum(w * a)
    bm = np.sum(w * b)
    ac = a - am
    bc = b - bm
    denom = math.sqrt(np.sum(w * ac**2) * np.sum(w * bc**2)) + 1e-12
    return float(np.sum(w * ac * bc) / denom)


## Mean-Anchor Disentanglement and Speed

This section compares latent speed, shape speed, volume-rate direction, residual leakage, and age-disease shape-space correlation.

In [6]:
# This cell computes mean-anchor latent speed and decoder-induced shape speed for every component.
# Interpret disease_raw as disease capacity and disease as the diagnosis-gated disease velocity.
GEOMETRY = {}
RESULTS = {}
metric_rows = []

for model_name, model in MODELS.items():
    print('Geometry:', model['label'])
    z = model['mean_z']
    geometry = surface_geometry(model['decoder'], z)
    velocities = component_vectors(model, z)
    GEOMETRY[model_name] = geometry
    RESULTS[model_name] = {}
    for component in COMPONENTS:
        print('  component:', component)
        result = surface_component_map(model, geometry, z, velocities[component])
        RESULTS[model_name][component] = result
        metric_rows.append({
            'model_key': model_name,
            'model': model['label'],
            'epoch': model['epoch'],
            'component': component,
            'median_abs_sdf_normal_alignment': geometry['median_normal_alignment'],
            'mesh_volume': geometry['signed_mesh_volume'],
            'mesh_watertight': geometry['mesh_is_watertight'],
            **result['metrics'],
        })

METRICS = pd.DataFrame(metric_rows)
for model_name, model in MODELS.items():
    weights = vertex_area_weights(GEOMETRY[model_name]['mesh'])
    corr = weighted_correlation(
        RESULTS[model_name]['age']['normal_speed'],
        RESULTS[model_name]['disease']['normal_speed'],
        weights,
    )
    METRICS.loc[METRICS['model_key'] == model_name, 'age_disease_shape_corr'] = corr
    total_rms = float(METRICS[(METRICS['model_key'] == model_name) & (METRICS['component'] == 'total')]['rms_normal_speed'].iloc[0])
    residual_rms = float(METRICS[(METRICS['model_key'] == model_name) & (METRICS['component'] == 'residual')]['rms_normal_speed'].iloc[0])
    METRICS.loc[METRICS['model_key'] == model_name, 'residual_shape_rms_fraction'] = residual_rms / (total_rms + 1e-12)

save_table(
    METRICS,
    '06_mean_anchor_speed_metrics',
    'Mean-anchor shape-space speed metrics',
    'Mean-anchor speed',
    'Latent speed, shape-space RMS speed, net volume rate, and leakage metrics for every component.',
)

summary = METRICS.pivot_table(index='model', columns='component', values='rms_normal_speed')
summary['residual_fraction'] = METRICS.groupby('model')['residual_shape_rms_fraction'].first()
summary['age_disease_corr'] = METRICS.groupby('model')['age_disease_shape_corr'].first()
save_table(
    summary.reset_index(),
    '07_mean_anchor_speed_summary',
    'Mean-anchor speed summary',
    'Mean-anchor speed',
    'Compact view of component RMS speed, residual fraction, and age-disease shape correlation.',
)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
plot_df = METRICS[METRICS['component'].isin(['age', 'disease', 'residual', 'total'])]
for component, sub in plot_df.groupby('component'):
    axes[0].plot(sub['model'], sub['latent_velocity_norm'], marker='o', label=component)
    axes[1].plot(sub['model'], sub['rms_normal_speed'], marker='o', label=component)
    axes[2].plot(sub['model'], sub['net_volume_rate'], marker='o', label=component)
for ax in axes:
    ax.tick_params(axis='x', rotation=25)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
axes[0].set_title('Latent velocity norm')
axes[1].set_title('Shape-space RMS normal speed')
axes[2].set_title('Net volume-rate direction')
fig.suptitle('Disentanglement check: a good model has disease in disease, age in age, and low residual shape speed')
fig.tight_layout()
save_matplotlib_figure(
    fig,
    '08_mean_anchor_velocity_decomposition',
    'Mean-anchor velocity decomposition',
    'Mean-anchor speed',
    'Compares latent norm, decoded surface RMS speed, and net volume-rate direction.',
)


Geometry: Subset 64/128/64 best
  component: age
  component: disease_raw
  component: disease
  component: residual
  component: total
Geometry: Subset 64/160/32 E2 best saved
  component: age
  component: disease_raw
  component: disease
  component: residual
  component: total
Geometry: Additive strong best
  component: age
  component: disease_raw
  component: disease
  component: residual
  component: total
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/06_mean_anchor_speed_metrics.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/07_mean_anchor_speed_summary.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/08_mean_anchor_velocity_decomposition.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/08_mean_anchor_velocity_decomposition.html')

## Surface Speed and INR Difference Maps

Rows are models and columns are components. The finite-step map is the professor-requested `INR(mean_z) - INR(mean_z + v*t)` visualization.

In [7]:
# This cell visualizes component-wise normal speed and INR finite-step differences on the decoded mean surface.
# Color means signed local surface motion: red/blue are opposite normal directions; near white means little effect.
def robust_symmetric_limit(arrays, quantile=0.98):
    vals = np.concatenate([np.ravel(a[np.isfinite(a)]) for a in arrays if np.size(a)])
    if vals.size == 0:
        return 1.0
    limit = float(np.quantile(np.abs(vals), quantile))
    return limit if limit > 0 else 1.0


def mesh_surface_trace(mesh, values, cmin, cmax, colorscale='RdBu_r', show_colorbar=False, title=''):
    vertices = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.faces)
    return go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        intensity=np.asarray(values),
        colorscale=colorscale,
        cmin=cmin, cmax=cmax,
        showscale=show_colorbar,
        colorbar=dict(title=title, thickness=18, len=0.72),
        opacity=1.0,
        flatshading=False,
        lighting=dict(ambient=0.58, diffuse=0.72, specular=0.18, roughness=0.55),
    )


def comparison_surface_figure(value_key, title, components=('age', 'disease', 'residual', 'total')):
    arrays = [RESULTS[name][component][value_key] for name in MODELS for component in components]
    limit = robust_symmetric_limit(arrays)
    fig = make_subplots(
        rows=len(MODELS), cols=len(components),
        specs=[[{'type': 'scene'}] * len(components) for _ in MODELS],
        row_titles=[MODELS[name]['label'] for name in MODELS],
        column_titles=[c.replace('_', ' ') for c in components],
        horizontal_spacing=0.01, vertical_spacing=0.02,
    )
    for row, name in enumerate(MODELS, start=1):
        mesh = GEOMETRY[name]['mesh']
        for col, component in enumerate(components, start=1):
            fig.add_trace(
                mesh_surface_trace(
                    mesh,
                    RESULTS[name][component][value_key],
                    -limit,
                    limit,
                    show_colorbar=(row == 1 and col == len(components)),
                    title=value_key.replace('_', ' '),
                ),
                row=row, col=col,
            )
    for scene_name in [k for k in fig.layout if str(k).startswith('scene')]:
        fig.layout[scene_name].update(
            aspectmode='data',
            xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
            camera=dict(eye=dict(x=1.55, y=1.55, z=1.05)),
        )
    fig.update_layout(
        title=f'{title}<br><sup>Shared color limit +/-{limit:.4g}; rows are models, columns are velocity components.</sup>',
        width=350 * len(components), height=330 * len(MODELS),
        margin=dict(l=130, r=30, t=90, b=20),
    )
    return fig

normal_speed_fig = comparison_surface_figure('normal_speed', 'Decoder-Jacobian normal speed on mean shape')
save_plotly_figure(
    normal_speed_fig,
    '09_mean_anchor_normal_speed_maps',
    'Mean-anchor normal speed maps',
    'Surface maps',
    'Decoder-Jacobian normal speed. Rows are models; columns are age, disease, residual, and total components.',
)

inr_difference_fig = comparison_surface_figure(
    'inr_difference',
    'Requested finite step: INR(mean_z) - INR(mean_z + v * DELTA_T)',
)
save_plotly_figure(
    inr_difference_fig,
    '10_mean_anchor_inr_difference_maps',
    'Mean-anchor INR difference maps',
    'Surface maps',
    'Finite-step visualization requested by professor: INR(mean_z) - INR(mean_z + v * DELTA_T).',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/09_mean_anchor_normal_speed_maps.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/10_mean_anchor_inr_difference_maps.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/10_mean_anchor_inr_difference_maps.html')

## SDF Difference Slices

These slices complement the surface maps by showing how each component changes the implicit field through the volume.

In [8]:
# This cell shows SDF-difference slices through the 3D volume.
# These slices are easier to inspect than a surface map when a component affects internal/external SDF regions differently.
def inr_difference_grid(model_name, component, delta_t=DELTA_T, resolution=GRID_RES):
    model = MODELS[model_name]
    z = model['mean_z']
    velocity = RESULTS[model_name][component]['velocity']
    _, xyz = regular_grid(resolution)
    base = decode_points(model['decoder'], z, xyz).reshape(resolution, resolution, resolution)
    moved = decode_points(model['decoder'], z + delta_t * velocity, xyz).reshape(resolution, resolution, resolution)
    return base - moved

components_to_show = ('age', 'disease', 'residual', 'total')
fig, axes = plt.subplots(len(MODELS), len(components_to_show), figsize=(4.2 * len(components_to_show), 3.8 * len(MODELS)))
mid = GRID_RES // 2
all_slices = []
for model_name in MODELS:
    for component in components_to_show:
        all_slices.append(inr_difference_grid(model_name, component)[:, :, mid])
limit = robust_symmetric_limit(all_slices)
for r, model_name in enumerate(MODELS):
    for c, component in enumerate(components_to_show):
        ax = axes[r, c]
        sl = inr_difference_grid(model_name, component)[:, :, mid]
        im = ax.imshow(sl.T, origin='lower', cmap='RdBu_r', vmin=-limit, vmax=limit)
        ax.contour(GEOMETRY[model_name]['sdf_grid'][:, :, mid].T, levels=[0], colors='black', linewidths=0.8)
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title(component)
        if c == 0:
            ax.set_ylabel(MODELS[model_name]['short_label'])
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.65, label='INR(z) - INR(z + v*dt)')
fig.suptitle('Orthogonal SDF-difference slice at z=0. Black contour is the decoded baseline surface.')
save_matplotlib_figure(
    fig,
    '11_mean_anchor_sdf_difference_slices',
    'Mean-anchor SDF difference slices',
    'SDF slices',
    'Orthogonal slice through INR(z) - INR(z + v*dt); black contour is the decoded baseline surface.',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/11_mean_anchor_sdf_difference_slices.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/11_mean_anchor_sdf_difference_slices.html')

## Part 1 Interpretation Guide

Use this notebook to answer these questions before looking at individual subjects:

- **Forecast model choice:** `64/160/32` loads epoch 1000 because it is the best saved E2 forecast checkpoint available. The CSV-only epoch 1500 result is shown but cannot be visualized unless that checkpoint is saved.
- **Disentanglement:** compare `age_disease_shape_corr` and `residual_shape_rms_fraction`. Lower residual fraction means the residual block is not explaining the main progression.
- **Disease branch:** compare `disease_raw` and `disease`. `disease_raw` shows the branch capacity; `disease` is after diagnosis gating.
- **Shape-space speed:** trust the surface maps only when `jvp_fd_relative_rmse` is small and mesh normal alignment is near 1.
- **Real speed:** the observed thickness/bump rates give the expected direction and age dependence. A model speed can be numerically large in latent space but biologically weak if it does not match these observed trends.


## Export Index
Run this after the cells above to refresh the indexed HTML report.


In [9]:
# Refresh the HTML index after running the output-saving cells above.
index_path = write_figure_index()
relative_index = index_path.relative_to(ROOT)
display(HTML(
    f"<p><strong>All Part 1 outputs:</strong> "
    f"<a href='{relative_index.as_posix()}' target='_blank'>{relative_index.as_posix()}</a></p>"
))
print('Saved Part 1 analysis:', OUTPUT_DIR)
print('Open this file for every Part 1 visualization/table:', index_path)


Figure index: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/index.html


Saved Part 1 analysis: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1
Open this file for every Part 1 visualization/table: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1/figures/index.html
